# Library `nn` — Panduan & Dokumentasi
Suatu **deep learning framework** yang dibuat _from scratch_ dari NumPy.
Notebook ini berperan sebagai **tutorial** dan **referensi** untuk semua _public_ API.

## Daftar Pustaka
1. [Instalasi & Import](#1-installation--imports)
2. [Tensor & Autograd](#2-core-concepts-tensor--autograd)
3. [Layers](#3-layers-reference)
4. [Model](#4-model-api-sequential)
5. [Pelatihan](#5-training-workflow)
6. [Contoh A — CNN](#6-example-a--cnn-digit-classification)
7. [Contoh B — SimpleRNN](#7-example-b--simplernn-digit-classification)
8. [Contoh C — LSTM](#8-example-c--lstm-digit-classification)
9. [Menyimpan & Memuat Bobot](#9-save--load-weights)
10. [Fitur Lanjut](#10-advanced-features)
11. [Referensi API](#11-api-quick-reference)

---
## 1. Instalasi & Imports
__Package__ ini diimplementasikan di `src/nn/`. Pastikan `src/` ada di direktori.

**Dependencies:** `numpy`, `scipy`, `tqdm` (untuk _verbose_ saja), `sklearn` (untuk demo saja).

In [3]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath('.'))

from nn import (
    Tensor, no_grad, Model,
    Dense, Conv2D, LocallyConnected2D,
    Flatten, MaxPooling2D, AveragePooling2D,
    GlobalMaxPooling2D, GlobalAveragePooling2D,
    Embedding, SimpleRNN, LSTM, RMSNorm,
    Adam, SGD, Input
)

---
## Tensor & Autograd
`Tensor` adalah komponen pokok dari library ini. Kelas ini membungkus array NumPy dan melengkapinya dengan operasi dasar
untuk _automatic differentiation_ (_autograd_).

### Properti
| Atribut | Deskripsi |
|---|---|
| `.data` | Data yang disimpan sebagai `np.ndarray` |
| `.grad` | Gradien yang terakumulasi setelah `.backward()` |
| `.requires_grad` | Jika `True`, tensor berpartisipasi dalam perhitungan gradien |

### Operasi
`+`, `-`, `*`, `/`, `**`, `@` (matmul), `sum()`, `mean()`, `reshape()`, `split()`, `concatenate()`, indexing/slicing — semua dengan fitur _autograd_.

In [5]:
a = Tensor([1.0, 2.0, 3.0], requires_grad=True)
b = Tensor([4.0, 5.0, 6.0], requires_grad=True)

c = (a * b).sum()  
print(f'c = {c.data}')

c.backward()
print(f'dc/da = {a.grad}  (should be [4, 5, 6])')
print(f'dc/db = {b.grad}  (should be [1, 2, 3])')

c = 32.0
dc/da = [4. 5. 6.]  (should be [4, 5, 6])
dc/db = [1. 2. 3.]  (should be [1, 2, 3])


### `no_grad()` Context Manager
Gunakan `no_grad()` untuk mematikan pelacakan gradien (seperti saat inferensi). Ini menghemat memori dan mempercepat komputasi.

In [6]:
with no_grad():
    x = Tensor([1.0, 2.0], requires_grad=True)
    y = x * 2
    print(f'Gradient tracking disabled: y has no graph -> {y._children}')

Gradient tracking disabled: y has no graph -> ()


---
## 3. Layers 
Semua layer mengikuti API yang konsisten: `layer.build(input_shape)` lalu `layer.forward(inputs)`.
Saat digunakan dalam `Model`, layer dibangun secara **otomatis**.

### Layer

| Layer | Konstruktor | Input Shape | Output Shape |
|---|---|---|---|
| `Input` | `Input(shape=(batch, ...))` | — | sama |
| `Dense` | `Dense(units, activation)` | `(B, features)` | `(B, units)` |
| `Conv2D` | `Conv2D(filters, kernel_size, stride, padding, activation)` | `(B, H, W, C)` | `(B, H', W', filters)` |
| `LocallyConnected2D` | `LocallyConnected2D(filters, kernel_size, stride, padding, activation)` | `(B, H, W, C)` | `(B, H', W', filters)` |
| `MaxPooling2D` | `MaxPooling2D(pool_size)` | `(B, H, W, C)` | `(B, H//p, W//p, C)` |
| `AveragePooling2D` | `AveragePooling2D(pool_size)` | `(B, H, W, C)` | `(B, H//p, W//p, C)` |
| `GlobalMaxPooling2D` | `GlobalMaxPooling2D()` | `(B, H, W, C)` | `(B, C)` |
| `GlobalAveragePooling2D` | `GlobalAveragePooling2D()` | `(B, H, W, C)` | `(B, C)` |
| `Flatten` | `Flatten()` | `(B, ...)` | `(B, prod(...))` |
| `Embedding` | `Embedding(vocab_size, embed_dim)` | `(B, seq_len)` | `(B, seq_len, embed_dim)` |
| `SimpleRNN` | `SimpleRNN(units, return_sequences)` | `(B, T, features)` | `(B, units)` atau `(B, T, units)` |
| `LSTM` | `LSTM(units, return_sequences)` | `(B, T, features)` | `(B, units)` atau `(B, T, units)` |
| `RMSNorm` | `RMSNorm()` | `(B, ...)` | sama |

### Aktivasi
`'linear'`, `'relu'`, `'sigmoid'`, `'tanh'`, `'softmax'`, `'swish'`, `'gelu'`

### Inisialisasi Bobot
`'zero'`, `'uniform'`, `'normal'`, `'xavier'`, `'he'`

---
## 4. Model 
Kelas `Model` mirip dirancang mirip dengan `Sequential` di Keras.

### Cara Membangun Model
**Opsi A: Masukkan layer pada konstruktor (direkomendasikan)**
```python
model = Model([
    Input(shape=(None, 784)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
], seed=42)
```

**Opsi B: Tambahkan layers satu-satu**
```python
model = Model()
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))
```

> **Catatan:** Jika menggunakan Opsi A dengan layer `Input`, model dibangun secara otomatis.
> Tanpa `Input`, pembangunan ditunda sampai `.fit()` atau `.build()` dipanggil.

In [7]:
demo = Model([
    Input(shape=(None, 4)),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
], seed=42)

demo.summary()


Layer                  Output Shape            Param #               
Input()                (None, 4)               0                     
Dense(relu)            (None, 8)               40                    
Dense(softmax)         (None, 3)               27                    
Total parameters: 67
Trainable parameters: 67
Non-trainable params: 0



---
## 5. Pelatihan

### Tahap 1: Compile
```python
model.compile(optimizer, loss, learning_rate=0.001)
```

| Optimizer | Loss |
|---|---|
| `'adam'` — Adaptive Moment Estimation | `'mse'` — Mean Squared Error |
| `'sgd'` — Stchastic Gradient Descent | `'bce'` — Binary Cross-Entropy |
| | `'cce'` — Categorical Cross-Entropy (one-hot) |
| | `'scce'` — Sparse Categorical Cross-Entropy (integer labels) |
#
> **Catatan:** Argumen optimizer dapat diisi suatu instansi Optimizer seperti `Adam(learning_rate=0.01)`.

### Tahap 2: Fit
```python
history = model.fit(X, y, epochs=10, batch_size=32, verbose=2, validation_data=(X_val, y_val))
```

| `verbose` | Perilaku |
|---|---|
| `0` | Tidak ada output |
| `1` | Satu baris per epoch |
| `2` | _Progress bar_ tqdm per epoch (default) |

### Tahap 3: Predict & Evaluate
```python
predictions = model.predict(X_test) 
loss_value  = model.evaluate(X_test, y_test) 
```

In [8]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

model_xor = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='tanh'),
    Dense(1, activation='sigmoid')
], seed=42)

model_xor.compile(optimizer='adam', loss='bce', learning_rate=0.05)
history = model_xor.fit(X_xor, y_xor, epochs=300, batch_size=4, verbose=0)

preds = model_xor.predict(X_xor)
print(f'Predictions: {np.round(preds.flatten(), 3)}')
print(f'Expected:    [0, 1, 1, 0]')
print(f'Final loss:  {history["train_loss"][-1]:.4f}')

Predictions: [0.    0.999 0.999 0.002]
Expected:    [0, 1, 1, 0]
Final loss:  0.0010


---
## 6. Contoh A — CNN 
Contoh menggunakan `sklearn.datasets.load_digits` (gambar _grayscale_ 8x8, 10 kelas).

**Arkitektur:** `Conv2D(8, 3)` -> `MaxPooling2D(2)` -> `Flatten` -> `Dense(32)` -> `Dense(10, softmax)`

In [9]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_all = digits.data / 16.0 
y_all = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=50, train_size=200, random_state=42, stratify=y_all
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}, Classes: {np.unique(y_train)}')

Train: (200, 64), Test: (50, 64), Classes: [0 1 2 3 4 5 6 7 8 9]


In [10]:
X_train_cnn = X_train.reshape(-1, 8, 8, 1)
X_test_cnn  = X_test.reshape(-1, 8, 8, 1)

model_cnn = Model([
    Input(shape=(None, 8, 8, 1)),
    Conv2D(8, kernel_size=3, padding=0, activation='relu'),
    MaxPooling2D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
], seed=42)

model_cnn.summary()
model_cnn.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_cnn = model_cnn.fit(
    X_train_cnn, y_train,
    epochs=15, batch_size=32, verbose=2,
    validation_data=(X_test_cnn, y_test)
)

preds_cnn = np.argmax(model_cnn.predict(X_test_cnn), axis=-1)
acc_cnn = np.mean(preds_cnn == y_test)
print(f'\nCNN Test Accuracy: {acc_cnn:.2%}')


Layer                  Output Shape            Param #               
Input()                (None, 8, 8, 1)         0                     
Conv2D(relu)           (None, 6, 6, 8)         80                    
MaxPooling2D()         (None, 3, 3, 8)         0                     
Flatten()              (None, 72)              0                     
Dense(relu)            (None, 32)              2,336                 
Dense(softmax)         (None, 10)              330                   
Total parameters: 2,746
Trainable parameters: 2,746
Non-trainable params: 0



Epoch 1/15:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 15/15: 100%|██████████| 7/7 [00:00<00:00, 253.06it/s, loss=0.2091, val_loss=0.4523]



CNN Test Accuracy: 92.00%


---
## 7. Contoh B — SimpleRNN 
Dataset yang sama, tetap dianggap **sekuens**: setiap gambar 8x8 menjadi 8 _time step_ berisi 8 fitur.

**Arkitektur:** `SimpleRNN(32)` -> `Dense(10, softmax)`

In [11]:
X_train_rnn = X_train.reshape(-1, 8, 8)
X_test_rnn  = X_test.reshape(-1, 8, 8)

model_rnn = Model([
    Input(shape=(None, 8, 8)),
    SimpleRNN(32),
    Dense(10, activation='softmax')
], seed=42)

model_rnn.summary()
model_rnn.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_rnn = model_rnn.fit(
    X_train_rnn, y_train,
    epochs=30, batch_size=32, verbose=2,
    validation_data=(X_test_rnn, y_test)
)

preds_rnn = np.argmax(model_rnn.predict(X_test_rnn), axis=-1)
acc_rnn = np.mean(preds_rnn == y_test)
print(f'\nSimpleRNN Test Accuracy: {acc_rnn:.2%}')


Layer                  Output Shape            Param #               
Input()                (None, 8, 8)            0                     
SimpleRNN()            (None, 32)              1,312                 
Dense(softmax)         (None, 10)              330                   
Total parameters: 1,642
Trainable parameters: 1,642
Non-trainable params: 0



Epoch 30/30: 100%|██████████| 7/7 [00:00<00:00, 80.59it/s, loss=0.0710, val_loss=0.3556]


SimpleRNN Test Accuracy: 84.00%


---
## 8. Contoh C — LSTM 
Dataset sama seperti SimpleRNN. LSTM menerapkan mekanisme _gates_ (forget, input, output).

**Arkitektur:** `LSTM(32)` -> `Dense(10, softmax)`

In [12]:
model_lstm = Model([
    Input(shape=(None, 8, 8)),
    LSTM(32),
    Dense(10, activation='softmax')
], seed=42)

model_lstm.summary()
model_lstm.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_lstm = model_lstm.fit(
    X_train_rnn, y_train,
    epochs=30, batch_size=32, verbose=2,
    validation_data=(X_test_rnn, y_test)
)

preds_lstm = np.argmax(model_lstm.predict(X_test_rnn), axis=-1)
acc_lstm = np.mean(preds_lstm == y_test)
print(f'\nLSTM Test Accuracy: {acc_lstm:.2%}')


Layer                  Output Shape            Param #               
Input()                (None, 8, 8)            0                     
LSTM()                 (None, 32)              5,248                 
Dense(softmax)         (None, 10)              330                   
Total parameters: 5,578
Trainable parameters: 5,578
Non-trainable params: 0



Epoch 30/30: 100%|██████████| 7/7 [00:00<00:00, 175.21it/s, loss=0.2415, val_loss=0.8804]



LSTM Test Accuracy: 70.00%


---
### Perbandingan Hasil

In [13]:
print('=' * 50)
print(f'{"Model":<15} {"Test Accuracy":>15} {"Final Loss":>12}')
print('=' * 50)
print(f'{"CNN":<15} {acc_cnn:>14.2%} {history_cnn["train_loss"][-1]:>12.4f}')
print(f'{"SimpleRNN":<15} {acc_rnn:>14.2%} {history_rnn["train_loss"][-1]:>12.4f}')
print(f'{"LSTM":<15} {acc_lstm:>14.2%} {history_lstm["train_loss"][-1]:>12.4f}')
print('=' * 50)

Model             Test Accuracy   Final Loss
CNN                     92.00%       0.2091
SimpleRNN               84.00%       0.0710
LSTM                    70.00%       0.2415


---
## 9. Menyimpan dan Memuat Bobot
Weights are saved as `.npz` files (NumPy compressed archives).

In [15]:
model_cnn.save('demo_cnn_weights')
print('Saved!')

model_cnn_loaded = Model([
    Input(shape=(None, 8, 8, 1)),
    Conv2D(8, kernel_size=3, padding=0, activation='relu'),
    MaxPooling2D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])
model_cnn_loaded.load('demo_cnn_weights.npz')

preds_original = model_cnn.predict(X_test_cnn)
preds_loaded   = model_cnn_loaded.predict(X_test_cnn)
print(f'Predictions match: {np.allclose(preds_original, preds_loaded)}')

os.remove('demo_cnn_weights.npz') 

Saved!
Predictions match: True


---
## 10. Fitur Lanjut
### Freeze / Unfreeze 
Layer dapat dibekukan agar parameternya tidak terdampak _updating_ gradien (misalnya untuk _transfer learning_).

In [16]:
model_freeze = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
], seed=42)

model_freeze.layers[1].freeze()
trainable = len(model_freeze.layers[1].parameters())
total     = len(model_freeze.layers[1].tensors())
print(f'Layer 1 trainable params: {trainable}')
print(f'Layer 1 total tensors:    {total}')

model_freeze.layers[1].unfreeze()
trainable = len(model_freeze.layers[1].parameters())
print(f'After unfreeze:           {trainable} trainable')

Layer 1 trainable params: 0
Layer 1 total tensors:    2
After unfreeze:           2 trainable


### Regularisasi (L1 / L2)
Gunakan argumen `l1_lambda` dan/atau `l2_lambda` pada konstruktor layer manapun. 

In [17]:
model_reg = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='relu', l1_lambda=0.01, l2_lambda=0.001),
    Dense(1, activation='sigmoid')
], seed=42)

model_reg.compile(optimizer='adam', loss='bce')
history = model_reg.fit(X_xor, y_xor, epochs=100, batch_size=4, verbose=0)
print(f'Final loss (with regularization): {history["train_loss"][-1]:.4f}')

Final loss (with regularization): 0.7544


### Instansi Optimizer Kustom
Daripada memberikan argumen string, optimizer dapat berupa objek `Optimizer` kustom.

In [18]:
from nn import Adam, SGD

opt = Adam(learning_rate=0.01)
model_custom = Model([
    Input(shape=(None, 2)),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_custom.compile(optimizer=opt, loss='bce')
history = model_custom.fit(X_xor, y_xor, epochs=200, batch_size=4, verbose=0)
print(f'Final loss: {history["train_loss"][-1]:.4f}')

Final loss: 0.0505


---
## 11. Referensi API

### Model
| Method | Deskripsi |
|---|---|
| `Model(layers, seed)` | Membuat model, dengan layer dan seed |
| `.add(layer)` | Tambah layer |
| `.build(input_shape)` | Inisialisasi bobot secara manual |
| `.compile(optimizer, loss, learning_rate)` | Atur optimizer dan fungsi loss |
| `.fit(X, y, epochs, batch_size, verbose, validation_data)` | Latih model |
| `.predict(X, batch_size)` | Jalankan _forward pass_, mengembalikan `np.ndarray` |
| `.evaluate(X, y, batch_size)` | Hitung loss pada data |
| `.summary()` | Print tabel layer/parameter |
| `.save(filepath)` | Simpan bobot pada `.npz` |
| `.load(filepath)` | Muat bobot dari `.npz` |
| `.parameters()` | Daftar semua objek `Tensor` _trainable_ |

### Tensor
| Method | Deskripsi |
|---|---|
| `Tensor(data, requires_grad)` | Membuat suatu tensor |
| `.backward()` | _Backpropagate_ gradien |
| `.data` | Data dalam NumPy array murni |
| `.grad` | Gradien yang terakumulasi |
| `Tensor.concatenate(tensors, axis)` | Konkatenasikan banyak tensor |
| `no_grad()` | _Context manager_ untuk mematikan _autograd_ |